# agent

> The turn: what the model is told, what it is allowed to do, what it did, and what that cost.

`Agent` is the whole harness in one object -- a routed conversation whose tools are the
host's capabilities. Around it sit the three things a person needs in order to trust it:
the activity feed that says what it is doing, the approval gate that stops a write until
they agree to it, and the briefing that tells the model where it is.

In [ ]:
#| default_exp agent

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail
from ramabana.tools import NullHost
from ramabana.testing import MemHost, FakeBackend, fake_agent

In [ ]:
#| export
import datetime, functools, json, re, threading, time, uuid
from dataclasses import dataclass, field
from ramabana.core import agent_err, available_models, JOBS, Routing, model_note
from ramabana.runtime import Usage, make_backend, Compactor, compact_notebook_context, notices_block
from ramabana.tools import (MAX_TOOL_CHARS, WRITE_TOOLS, Registry, clip, discover, err, failed,
                            find, load, skill_index, subagent_tools, tools_for)

## The activity feed

What the agent is doing, as a person would describe it. The summaries are written per
tool rather than rendered from `name(args)`, because a generic render is unreadable at a
glance -- and a glance is the only way this list is ever read.

In [ ]:
#| export
MAX_DETAIL = 4000     # chars of a tool result kept for the fold
MAX_ACTS = 500        # a very long turn should not grow without bound
MAX_CHECKPOINTS = 20  # turn boundaries kept for `fork`; each one is a whole conversation
POLL_EVERY = 900      # seconds between automatic `Host.poll` ticks; a turn is what triggers one

#: Characters of the open folders `changes()` will hold in order to watch a shell command.
#: A command names no files, so the only way to know what it moved is to have read them
#: first; past this much text that costs more than the answer is worth.
SHELL_SNAPSHOT = 32_000_000

ICONS = {'search': '🔍', 'view': '📄', 'edit': '✏️', 'web': '🌐', 'run': '▶️',
         'skill': '📚', 'delegate': '🤝', 'memory': '🧠', 'watch': '⏰', 'cart': '🛒',
         'tool': '🔧'}

_KIND = {
    'search_code': 'search', 'similar_code': 'search', 'outline': 'search', 'list_files': 'search',
    'grep': 'search', 'ls': 'search',
    'view_file': 'view', 'notebook_cells': 'view', 'view_cell': 'view', 'read_terminal': 'view',
    'edit_file': 'edit', 'replace_text': 'edit', 'create_file': 'edit', 'edit_cell': 'edit',
    'add_cell': 'edit',
    'web_search': 'web', 'read_url': 'web', 'research': 'web',
    'run_python': 'run', 'list_vars': 'run', 'run_shell': 'run',
    'read_skill': 'skill', 'create_skill': 'skill',
    'delegate_search': 'delegate', 'delegate_parallel': 'delegate',
    'inspect_python': 'run',
    'memory_search': 'memory', 'memory_tree': 'memory', 'memory_read': 'memory',
    'memory_topics': 'memory', 'memory_forget': 'memory', 'remember': 'memory',
    'set_reminder': 'watch', 'watch_url': 'watch', 'list_watches': 'watch',
    'cancel_watch': 'watch', 'poll_watches': 'watch',
    'cart_stores': 'cart', 'cart_open': 'cart', 'cart_find': 'cart',
    'cart_add': 'cart', 'cart_show': 'cart', 'cart_remove': 'cart',
}

#: Tools whose useful summary is the same words every time: they take nothing worth showing.
_LABEL = {'list_vars': 'List variables', 'read_terminal': 'Read terminal',
          'list_watches': 'List watches', 'poll_watches': 'Poll watches',
          'memory_topics': 'Map remembered topics', 'cart_stores': 'List stores',
          'cart_show': 'Read the trolley'}

def _s(v, n=90):
    "One line of a value, short enough to sit in a list."
    t = ' '.join(str(v or '').split())
    return t if len(t) <= n else t[:n - 1] + '…'

In [ ]:
#| export
def summarise(tool, args):
    """The imperative one-liner for a call: what a person would say they just did.

    Per tool rather than a generic `name(args)` render, because the useful summary is
    different every time and the generic one is unreadable at a glance -- which is the
    only way this list is ever read.
    """
    a = args if isinstance(args, dict) else {}
    p, q = a.get('path', ''), a.get('query', '')
    if tool in _LABEL:           return _LABEL[tool]
    if tool == 'search_code':    return f'Search {_s(q)}'
    if tool == 'similar_code':   return f'Similar to {p}:{a.get("line", 1)}'
    if tool == 'outline':        return f'Outline {p}'
    if tool == 'list_files':     return f'List files {_s(a.get("pattern", "")) or "(all)"}'
    if tool == 'grep':
        where = f' in {a["path_filter"]}' if a.get('path_filter') else ''
        return f'Grep {_s(a.get("pattern", ""))}{where}'
    if tool == 'ls':             return f'List {p or "(open folders)"}'
    if tool == 'view_file':
        rng = f':{a["start"]}-{a["end"]}' if a.get('start') or a.get('end') else ''
        return f'View {p}{rng}'
    if tool == 'edit_file':      return f'Edit {p}'
    if tool == 'replace_text':   return f'Edit {p}'
    if tool == 'create_file':    return f'Create {p}'
    if tool == 'notebook_cells': return f'Cells of {p}'
    if tool == 'view_cell':      return f'View {p} cell {a.get("cell_id", "?")}'
    if tool == 'edit_cell':      return f'Edit {p} cell {a.get("cell_id", "?")}'
    if tool == 'add_cell':       return f'Add {a.get("cell_type", "code")} cell to {p}'
    if tool == 'web_search':     return f'Web search: {_s(q)}'
    if tool == 'read_url':       return f'Web fetch: {_s(a.get("url", ""), 120)}'
    if tool == 'research':       return f'Research: {_s(q)}'
    if tool == 'run_python':     return f'Run python: {_s((a.get("code", "") or "").strip().splitlines()[0] if a.get("code") else "")}'
    if tool == 'inspect_python':
        sc = a.get('scope') or 'isolated'
        body = _s((a.get('code', '') or '').strip().splitlines()[0] if a.get('code') else '')
        return f'Inspect: {body}' + ('' if sc == 'isolated' else f'  [{sc}]')
    if tool == 'run_shell':      return f'Run {_s(a.get("command", ""), 110)}'
    if tool == 'read_skill':     return f'Read skill {a.get("name", "?")}'
    if tool == 'create_skill':   return f'Create skill {a.get("name", "?")}'
    if tool == 'memory_search':  return f'Memory search: {_s(q)}'
    if tool == 'memory_tree':    return f'Memory tree {_s(a.get("document", "")) or "(everything)"}'
    if tool == 'memory_read':    return f'Memory read {a.get("node_id", "?")}'
    if tool == 'memory_forget':  return f'Forget {a.get("doc_id", "?")}'
    if tool == 'remember':       return f'Remember {_s(a.get("title") or a.get("text", ""))}'
    if tool == 'set_reminder':   return f'Remind every {a.get("every", "1w")}: {_s(a.get("text", ""))}'
    if tool == 'watch_url':      return f'Watch {_s(a.get("url", ""), 100)} every {a.get("every", "1d")}'
    if tool == 'cancel_watch':   return f'Cancel watch {a.get("watch_id", "?")}'
    if tool == 'cart_open':      return f'Shop at {_s(a.get("url", ""), 100)}'
    if tool == 'cart_find':      return f'Shop search: {_s(q)}'
    if tool == 'cart_add':       return f'Add to trolley: {a.get("qty", 1)} x {_s(a.get("item", ""))}'
    if tool == 'cart_remove':    return f'Remove from trolley: {_s(a.get("line", ""))}'
    if tool == 'delegate_search': return f'Delegate: {_s(a.get("question", ""), 120)}'
    if tool == 'delegate_parallel':
        try:
            import json as _j
            qs = _j.loads(a.get('questions') or '[]')
            return f'Delegate {len(qs)} questions in parallel: ' + _s('; '.join(qs), 110)
        except Exception: return f'Delegate in parallel: {_s(a.get("questions", ""), 110)}'
    inner = ', '.join(f'{k}={_s(v, 30)!r}' for k, v in a.items())
    return f'{tool}({inner})'

In [ ]:
[summarise(t, a) for t, a in [
    ('search_code', {'query': 'where is compaction triggered'}),
    ('view_file', {'path': 'nbs/01_runtime.ipynb', 'start': 40, 'end': 80}),
    ('edit_cell', {'path': 'nbs/00_core.ipynb', 'cell_id': '451586c0'}),
    ('inspect_python', {'code': 'list(df.columns)', 'scope': 'overlay'}),
    ('delegate_parallel', {'questions': '["what imports fastllm?", "where is threshold used?"]'}),
]]

['Search where is compaction triggered',
 'View nbs/01_runtime.ipynb:40-80',
 'Edit nbs/00_core.ipynb cell 451586c0',
 'Inspect: list(df.columns)  [overlay]',
 'Delegate 2 questions in parallel: what imports fastllm?; where is threshold used?']

A tool nobody wrote a summary for still gets a usable line, which is what keeps an
extension's tool from being invisible in the feed.

In [ ]:
summarise('word_count', {'path': '/proj/a.py'})

"word_count(path='/proj/a.py')"

An `Act` is one call from the moment it starts. It exists *before* the result does, which
is the point: a frontend can show `⏳ Web fetch: …` while the fetch is happening.

In [ ]:
#| export
@dataclass
class Act:
    "One tool call, from the moment it starts to whatever it returned."
    tool: str
    args: dict = field(default_factory=dict)
    summary: str = ''
    detail: str = ''
    ok: bool = True
    done: bool = False
    secs: float = 0.0
    started: float = field(default_factory=time.time)
    id: str = field(default_factory=lambda: uuid.uuid4().hex[:12])
    turn_id: str = ''
    revision: int = 0
    branch_id: str = 'main'
    parent_action_id: str = ''
    state: str = 'running'

    @property
    def kind(self): return _KIND.get(self.tool, 'tool')

    @property
    def icon(self): return ICONS.get(self.kind, ICONS['tool'])

    def finish(self, out, ok=True):
        self.detail = _clip(out)
        self.ok, self.done, self.state = ok, True, ('complete' if ok else 'failed')
        self.secs = round(time.time() - self.started, 2)
        return self

    def line(self):
        "The single line: icon, what it did, and how long -- or an hourglass while it runs."
        if not self.done: return f'⏳ {self.summary}'
        tail = f'  ({self.secs:.1f}s)' if self.secs >= 0.5 else ''
        return f'{"" if self.ok else "⚠️ "}{self.icon} {self.summary}{tail}'

    def md(self, fold=True):
        "This call as markdown: the line, with the result folded under it when there is one."
        if not (self.detail and fold): return f'- {self.line()}'
        body = self.detail.replace('```', '`​``')     # a fence in a result must not end ours
        return (f'- {self.line()}\n\n'
                f'  <details><summary>result</summary>\n\n  ```\n{_indent(body)}\n  ```\n\n  </details>')

    def dict(self):
        return {'id': self.id, 'action_id': self.id, 'turn_id': self.turn_id,
                'revision': self.revision, 'branch_id': self.branch_id,
                'parent_action_id': self.parent_action_id, 'state': self.state,
                'tool': self.tool, 'kind': self.kind, 'icon': self.icon,
                'summary': self.summary, 'line': self.line(), 'detail': self.detail,
                'ok': self.ok, 'done': self.done, 'secs': self.secs,
                'args': {k: _s(v, 300) for k, v in (self.args or {}).items()}}


def _clip(out, n=MAX_DETAIL):
    s = '' if out is None else str(out)
    return s if len(s) <= n else s[:n] + f'\n…[{len(s)-n} more chars]'


def _indent(s, pad='  '):
    return '\n'.join(pad + l for l in s.splitlines())

In [ ]:
a = Act('read_url', {'url': 'https://nbdev.fast.ai/api/export.html'}, summarise('read_url', {'url': 'https://nbdev.fast.ai/api/export.html'}))
a.kind, a.icon, a.line()

('web', '🌐', '⏳ Web fetch: https://nbdev.fast.ai/api/export.html')

Finished, it carries the outcome, the duration and the result -- and `state` moves to
`complete` or `failed`, which is what a UI colours on.

In [ ]:
a.finish('# Exporting a notebook to a library\n\nnb_export(...)')
a.line(), a.state

('🌐 Web fetch: https://nbdev.fast.ai/api/export.html', 'complete')

As markdown, the result is folded under the line. An answer buried under thirty tool calls
is an answer nobody reads, so the working is one click away rather than in the way.

In [ ]:
print(a.md())

- 🌐 Web fetch: https://nbdev.fast.ai/api/export.html

  <details><summary>result</summary>

  ```
  # Exporting a notebook to a library
  
  nb_export(...)
  ```

  </details>


In [ ]:
bad = Act('edit_file', {'path': 'a.py'}, 'Edit a.py').finish('edit failed: stale address', ok=False)
test_eq(bad.state, 'failed')
bad.line()

'⚠️ ✏️ Edit a.py'

`Activity` is the stream, and the hook a frontend hangs a redraw on. `on_change` fires
twice per call -- once at the start and once at the end -- which is the entire difference
between a UI that looks alive and one that looks stuck.

In [ ]:
#| export
class Activity:
    """The stream of calls for a session, and the hook a frontend hangs a redraw on.

    `on_change` fires twice per call -- once when it starts and once when it finishes --
    so a frontend can show `⏳ Web fetch: …` while the fetch is happening rather than only
    afterwards. That is the entire difference between a UI that looks alive and one that
    looks stuck, and it is why this is not simply a list appended to after the fact.

    Thread-safe because tools run on the model's worker thread and frontends read from
    theirs. The lock is around list mutation only; `on_change` is called outside it, so a
    slow frontend cannot stall the model.
    """

    def __init__(self, on_change=None, max_acts=MAX_ACTS):
        self.acts, self.on_change, self.max_acts = [], on_change, max_acts
        self._lock = threading.Lock()
        self._mark = 0
        self.turn_id = ''

    def __len__(self): return len(self.acts)

    def start(self, tool, args, action_id='', turn_id='', revision=0, branch_id='main', parent_action_id=''):
        a = Act(tool=tool, args=dict(args or {}), summary=summarise(tool, args),
                id=action_id or uuid.uuid4().hex[:12], turn_id=turn_id or self.turn_id,
                revision=int(revision or 0), branch_id=branch_id or 'main',
                parent_action_id=parent_action_id or '')
        with self._lock:
            self.acts.append(a)
            if len(self.acts) > self.max_acts: del self.acts[:-self.max_acts]
        self._changed(a)
        return a

    def finish(self, act, out, ok=True):
        act.finish(out, ok)
        self._changed(act)
        return act

    def _changed(self, act):
        if not self.on_change: return
        try: self.on_change(act)
        except Exception: pass

    # -- reading it ----------------------------------------------------------
    def mark(self, turn_id=''):
        "Remember where the stream is now and bind new actions to one durable turn id."
        self._mark = len(self.acts)
        if turn_id: self.turn_id = str(turn_id)
        return self._mark

    def since(self, mark=None):
        return self.acts[(self._mark if mark is None else mark):]

    def rows(self, n=None, mark=None):
        "The stream as dicts, for a frontend. `mark` limits it to one turn."
        acts = self.since(mark) if mark is not None else self.acts
        return [a.dict() for a in (acts[-n:] if n else acts)]

    def md(self, mark=None, fold=True, title='what I did'):
        """The stream as markdown, for saving into a notebook cell.

        Wrapped in a `<details>` of its own so a reply is readable at a glance and the
        working is one click away -- an answer buried under thirty tool calls is an answer
        nobody reads.
        """
        acts = self.since(mark) if mark is not None else self.acts
        if not acts: return ''
        body = '\n'.join(a.md(fold) for a in acts)
        return f'<details><summary>{title} ({len(acts)} steps)</summary>\n\n{body}\n\n</details>'

    def lines(self, mark=None):
        "Just the summary lines, for a status pane with no room for folds."
        return [a.line() for a in (self.since(mark) if mark is not None else self.acts)]

In [ ]:
seen = []
feed = Activity(on_change=lambda act: seen.append((act.tool, act.done)))
act = feed.start('search_code', {'query': 'compaction'})
feed.finish(act, 'nbs/01_runtime.ipynb:88  threshold')
seen

[('search_code', False), ('search_code', True)]

A slow or broken frontend cannot stall the model: `on_change` is called outside the lock,
and its exceptions are swallowed.

In [ ]:
noisy = Activity(on_change=lambda act: 1/0)
noisy.finish(noisy.start('outline', {'path': 'a.py'}), 'ok')
len(noisy)

1

`mark` remembers where a turn began, so the same stream serves the session view and the
one-turn fold.

In [ ]:
feed.mark('turn_000002')
feed.finish(feed.start('view_file', {'path': 'a.py'}), 'def a(): return 1')
feed.lines(), feed.lines(mark=feed._mark)

(['🔍 Search compaction', '📄 View a.py'], ['📄 View a.py'])

In [ ]:
test_eq(len(feed.rows()), 2)
test_eq(len(feed.rows(mark=feed._mark)), 1)
feed.rows(mark=feed._mark)[0]['turn_id']

'turn_000002'

## Approvals

A write is put in front of a person before it happens. Everything here exists to make that
exchange honest: what the call would actually do, one request at a time, and a refusal that
carries the reason back to the model rather than the word "denied".

In [ ]:
#| export
DENIED = 'Denied by human operator'

DFLT_TIMEOUT = 300      # seconds to wait for a person before giving up on one request
MAX_PREVIEW = 2000      # chars of "what would change"; a person will not read more


def _args(args):
    "A tool call's arguments as a dict, whether the model sent a dict or a JSON string."
    if isinstance(args, str):
        try: args = json.loads(args)
        except Exception: return {}
    return args if isinstance(args, dict) else {}


def _tc(tool_call):
    "`(name, args)` from a tool call in either backend's shape."
    if hasattr(tool_call, 'name'): return tool_call.name, _args(getattr(tool_call, 'arguments', {}))
    fn = (tool_call or {}).get('function', {}) if isinstance(tool_call, dict) else {}
    return fn.get('name', '?'), _args(fn.get('arguments', {}))

In [ ]:
#| export
def _fmt_cmds(commands):
    "exhash commands as one readable block, rather than as a JSON blob nobody reads."
    try:
        cmds = json.loads(commands) if isinstance(commands, str) else commands
        if not isinstance(cmds, list): raise ValueError
    except Exception:
        return str(commands)[:MAX_PREVIEW]
    out = []
    for c in cmds:
        if not isinstance(c, (list, tuple)) or not c: out.append(str(c)); continue
        addr, op, rest = c[0], (c[1] if len(c) > 1 else ''), list(c[2:])
        body = '\n'.join('    ' + str(r).replace('\n', '\n    ') for r in rest)
        out.append(f'{addr} {op}' + (f'\n{body}' if body else ''))
    return '\n'.join(out)


def preview_for(name, args, host=None):
    """What this call would actually do, as text a person can read in a couple of seconds.

    Per tool rather than generic, because the useful preview is different every time: for
    an edit it is the commands, for a new file it is the head of the file, for a cell it is
    which cell. Anything unrecognised falls back to the arguments, which is still better
    than the tool's name alone.
    """
    p = args.get('path', '')
    if name == 'edit_file':   return f'{p}\n\n{_fmt_cmds(args.get("commands", ""))}'[:MAX_PREVIEW]
    if name == 'edit_cell':   return f'{p} cell {args.get("cell_id","?")}\n\n{_fmt_cmds(args.get("commands",""))}'[:MAX_PREVIEW]
    if name == 'create_file':
        text, exists = args.get('text', ''), False
        try: exists = bool(host and host.check(p).exists())
        except Exception: pass
        head = f'{p}  ({"OVERWRITES an existing file" if exists else "new file"}, {len(text)} chars)\n\n'
        return (head + text)[:MAX_PREVIEW]
    if name == 'add_cell':
        return f'{p}  (new {args.get("cell_type","code")} cell at {args.get("index",-1)})\n\n{args.get("source","")}'[:MAX_PREVIEW]
    if name == 'run_python':  return str(args.get('code', ''))[:MAX_PREVIEW]
    return json.dumps(args, indent=2, default=str)[:MAX_PREVIEW]


def _summary(name, args):
    "The one-line version, for a status bar or a footer."
    if p := args.get('path'): return f'{name} → {p}'
    return f'{name}({", ".join(sorted(args))})'

Either backend's tool-call shape reduces to `(name, args)`, and a JSON argument string is
parsed rather than shown raw.

In [ ]:
_tc({'function': {'name': 'edit_file', 'arguments': '{"path": "a.py", "commands": "[]"}'}})

('edit_file', {'path': 'a.py', 'commands': '[]'})

The preview is per tool, because the useful preview is different every time: for an edit it
is the commands, for a new file it is the head of the file.

In [ ]:
print(preview_for('edit_file', {'path': 'a.py', 'commands': '[["2|ab12|", "s", "return 1", "return 2"]]'}))

a.py

2|ab12| s
    return 1
    return 2


In [ ]:
print(preview_for('create_file', {'path': 'new.py', 'text': 'def f(): pass\n'}, host=NullHost(['/proj'])))

new.py  (new file, 14 chars)

def f(): pass



Anything unrecognised falls back to the arguments, which is still better than a tool name
on its own.

In [ ]:
print(preview_for('word_count', {'path': '/proj/a.py'}))

{
  "path": "/proj/a.py"
}


An `Ask` is one request and the person's answer to it. It is truthy exactly when approved,
so an `Ask` *is* the decision -- both backends' `if not approve(tc)` keeps working, and the
reason travels with it.

In [ ]:
#| export
@dataclass
class Ask:
    """One request, and the person's answer to it.

    `answer` is None while it is pending, which is also what the frontends poll on. The
    `Event` is what the model's thread is sitting on; nothing outside this module should
    touch it, and `answer()` is the only thing that sets it.
    """
    tool: str
    args: dict = field(default_factory=dict)
    summary: str = ''
    preview: str = ''
    id: str = field(default_factory=lambda: uuid.uuid4().hex[:8])
    answer: bool = None
    note: str = ''
    asked: float = field(default_factory=time.time)
    _done: threading.Event = field(default_factory=threading.Event, repr=False, compare=False)

    @property
    def pending(self): return self.answer is None

    def __bool__(self):
        """Truthy exactly when approved, so an `Ask` *is* the approval decision.

        This is what lets `Approvals.gate` return the whole object instead of a bool.
        Rishi's `if not ok` and fastllm's guard both keep working, and the reason the user
        gave travels with the decision to whoever formats the refusal.
        """
        return self.answer is True

    def dict(self):
        return {'id': self.id, 'tool': self.tool, 'summary': self.summary, 'preview': self.preview,
                'answer': self.answer, 'note': self.note, 'pending': self.pending}

    def resolve(self, ok, note=''):
        self.answer, self.note = bool(ok), note or ''
        self._done.set()
        return self

    def wait(self, timeout):
        "Block until answered or `timeout` seconds pass. Returns whether it was answered."
        return self._done.wait(timeout)

    def reply(self):
        """What the model is told. A refusal with a reason is the whole point of this module.

        An approval with a note carries it too: "yes, but keep the docstring" is guidance
        the model should have while it is making the edit, not after.
        """
        if self.answer: return f'Approved by the user. Note from the user: {self.note}' if self.note else None
        return f'{DENIED}. Reason given: {self.note}' if self.note else DENIED

In [ ]:
#| export
def ask_md(ask):
    "An approval request as markdown -- what a person reads, and what is saved in the notebook."
    body = ask.preview.strip()
    fence = '```\n' + body + '\n```\n\n' if body else ''
    return (f'**🔐 approval needed — `{ask.tool}`**\n\n{ask.summary}\n\n{fence}'
            'Approve, or refuse with a reason — the reason goes back to the model.')

def answer_md(ask):
    "The person's half of the exchange, in the same voice."
    head = '**✅ approved**' if ask.answer else '**⛔ refused**'
    return f'{head} — `{ask.tool}`' + (f'\n\n{ask.note}' if ask.note else '')

In [ ]:
ask = Ask('edit_file', {'path': 'a.py'}, 'edit_file → a.py', 'a.py\n\n2|ab12| s')
ask.pending, bool(ask)

(True, False)

A refusal with a reason is the whole point: the model is told why, and can change the
approach instead of retrying the same edit.

In [ ]:
ask.resolve(False, 'keep the docstring')
bool(ask), ask.reply()

(False, 'Denied by human operator. Reason given: keep the docstring')

An approval with a note carries it too -- "yes, but keep the docstring" is guidance the
model should have *while* making the edit.

In [ ]:
test_eq(Ask('edit_file').resolve(True).reply(), None)          # a plain yes needs no words
Ask('edit_file').resolve(True, 'but keep the docstring').reply()

'Approved by the user. Note from the user: but keep the docstring'

In [ ]:
print(ask_md(Ask('create_file', {'path': 'new.py'}, 'create_file → new.py', 'new.py  (new file, 15 chars)')))

**🔐 approval needed — `create_file`**

create_file → new.py

```
new.py  (new file, 15 chars)
```

Approve, or refuse with a reason — the reason goes back to the model.


In [ ]:
answer_md(ask)

'**⛔ refused** — `edit_file`\n\nkeep the docstring'

`Approvals` is the queue of one and the thread handshake behind it. One at a time on
purpose: a model that wants to edit four files should be answered four times, because "yes
to all of that" is exactly the answer people give when they have not read any of it.

In [ ]:
#| export
class Approvals:
    """The queue of one, and the thread handshake behind it.

    One at a time on purpose. A model that wants to edit four files should be answered
    four times, because "yes to all of that" is exactly the answer people give when they
    have not read any of it -- and the tools run sequentially on the local backend anyway.
    `mode` is where a bulk answer belongs instead: set it to `'auto'` for a session where
    the user has decided to stop being asked, and it is a deliberate act rather than a
    slip of the return key.

    `listeners` is not decoration. If no frontend has registered, nobody will ever answer,
    and `gate` would block the model's worker thread until the timeout for no reason. With
    zero listeners it refuses immediately and says why, which is a bad outcome that is at
    least a fast and legible one.
    """

    def __init__(self,
                 tools=(),                  # tool names that need approval; everything else runs
                 mode='ask',                # 'ask' | 'auto' (approve everything) | 'off' (refuse everything)
                 timeout=DFLT_TIMEOUT,
                 host=None,                 # for previews that need to look at disk
                 on_ask=None,               # called with the `Ask` when one is raised
                 on_answer=None):           # called with the `Ask` when it is answered
        self.tools, self.mode, self.timeout, self.host = frozenset(tools), mode, timeout, host
        # `on_ask`/`on_answer` are the *application's* recorder -- in leela, the thing that
        # writes the exchange into the notebook. Frontends register through `listen`
        # instead, so a second frontend opening does not silently unhook the first, or the
        # recorder. Both halves of an exchange must reach every one of them.
        self.on_ask, self.on_answer = on_ask, on_answer
        self.current = None                 # the `Ask` in flight, or None
        self.history = []                   # every `Ask` this session, answered or not
        self._watchers = []                 # (on_ask, on_answer) per registered frontend
        self._lock = threading.Lock()

    # -- the frontend side ---------------------------------------------------
    @property
    def listeners(self): return len(self._watchers)

    def listen(self, on_ask=None, on_answer=None):
        """Register a frontend, and how to reach it. Returns a callable that unregisters it.

        Registering is what makes asking possible at all: with nobody listening, `request`
        refuses immediately rather than parking the model's worker thread until the timeout
        for an answer that was never going to come.
        """
        w = (on_ask, on_answer)
        with self._lock: self._watchers.append(w)
        done = [False]
        def stop():
            if done[0]: return
            done[0] = True
            with self._lock:
                if w in self._watchers: self._watchers.remove(w)
        return stop

    def _notify(self, which, a):
        "Call the recorder and every watcher, swallowing failures so one bad frontend cannot block a turn."
        fns = [getattr(self, f'on_{which}')] + [w[0 if which == 'ask' else 1] for w in list(self._watchers)]
        for f in fns:
            if not f: continue
            try: f(a)
            except Exception: pass

    @property
    def pending(self):
        "The request waiting for an answer, or None. What both frontends poll."
        a = self.current
        return a if (a is not None and a.pending) else None

    def answer(self, id, ok, note='', session=False):
        """Answer the pending request; optionally approve all later writes this session.

        ``session`` only has meaning for an approval. A refusal can carry guidance, but
        must never silently turn the policy off for later, unrelated requests.
        """
        a = self.current
        if a is None or a.id != id or not a.pending: return None
        if ok and session: self.mode = 'auto'
        # Record and notify before waking the model thread. Notebook recorders therefore
        # finish inserting the answer cell before a completed turn can repaint or save the
        # document; `Ask.resolve` used to set the event first, making this a race.
        a.answer, a.note = bool(ok), note or ''
        if ok and session and not a.note: a.note = 'approved for the rest of this session'
        self._notify('answer', a)
        a._done.set()
        return a

    def _decided(self, a, ok, note):
        """Resolve without asking anybody, and still tell the recorder.

        A refusal nobody hears about reaches the user as a tool failure with no explanation,
        which is the one outcome this class exists to prevent.
        """
        a.resolve(ok, note)
        self._notify('answer', a)
        return a

    def cancel_all(self, note='the turn was cancelled'):
        "Refuse anything in flight, so a stopped turn does not leave a worker thread parked."
        a = self.pending
        if a is not None: self.answer(a.id, False, note)

    # -- the model side ------------------------------------------------------
    def gate(self, tool_call):
        """The `approve(tool_call)` both backends call. Blocks the model's thread.

        Returns the `Ask`, not a bool: it is falsy when refused, so every existing
        `if not approve(tc)` still reads correctly, and it carries `reply()` so the reason
        the person gave can reach the model instead of being flattened to "denied".
        """
        return self.request(*_tc(tool_call))

    def request(self, name, args, force=False, timeout=None):
        """Raise one request and wait for it. Returns the resolved `Ask`.

        Returned rather than a bare bool because the caller needs `reply()` -- the reason
        the user gave is the part worth carrying back, and a bool has nowhere to put it.
        `force` lets an external client explicitly request a browser decision even when its
        tool name is not part of Leela's own write-tool policy.
        """
        a = Ask(tool=name, args=args, summary=_summary(name, args), preview=preview_for(name, args, self.host))
        self.history.append(a)
        if not force and name not in self.tools: return a.resolve(True)
        if self.mode == 'auto': return a.resolve(True)
        if self.mode == 'off': return self._decided(a, False, 'approval is switched off for this session')
        if self.listeners < 1:
            return self._decided(a, False, 'nothing is listening for approvals, so this could not be asked')
        self.current = a
        self._notify('ask', a)
        wait_for = self.timeout if timeout is None else timeout
        if not a.wait(wait_for):
            a.resolve(False, f'no answer after {wait_for}s')
            self._notify('answer', a)
        return a

With nothing listening it refuses immediately and says so. Otherwise `gate` would park the
model's worker thread until the timeout waiting for an answer that was never going to come.

In [ ]:
gate = Approvals(tools=WRITE_TOOLS)
gate.listeners, gate.request('edit_file', {'path': 'a.py'}).reply()

(0,
 'Denied by human operator. Reason given: nothing is listening for approvals, so this could not be asked')

A frontend registers, and answers. Here it answers inside the notification, which is
exactly what a UI does a moment later on a human's behalf.

In [ ]:
stop = gate.listen(on_ask=lambda a: gate.answer(a.id, False, 'rewrite it as a patch instead'))
decision = gate.request('edit_file', {'path': 'a.py', 'commands': '[]'})
bool(decision), decision.reply()

(False,
 'Denied by human operator. Reason given: rewrite it as a patch instead')

A tool that is not in the policy is approved without anyone being asked, which is how read
tools stay instant.

In [ ]:
test_eq(bool(gate.request('search_code', {'query': 'x'})), True)
test_eq(gate.pending, None)
len(gate.history)

3

The two bulk answers are deliberate acts rather than a slip of the return key: `'auto'`
approves everything for the session, `'off'` refuses everything and says which.

In [ ]:
Approvals(tools=WRITE_TOOLS, mode='auto').request('edit_file', {}).reply(), Approvals(tools=WRITE_TOOLS, mode='off').request('edit_file', {}).reply()

(None,
 'Denied by human operator. Reason given: approval is switched off for this session')

`session=True` on an approval is what turns the policy off, and it records that it did.

In [ ]:
stop()
gate.listen(on_ask=lambda a: gate.answer(a.id, True, session=True))
first = gate.request('edit_file', {'path': 'a.py'})
gate.mode, first.note

('auto', 'approved for the rest of this session')

A timeout is a refusal that explains itself, and a cancelled turn releases whoever was
waiting rather than leaving a worker thread parked.

In [ ]:
slow = Approvals(tools=WRITE_TOOLS, timeout=0.01)
slow.listen(on_ask=lambda a: None)
slow.request('edit_file', {'path': 'a.py'}).reply()

'Denied by human operator. Reason given: no answer after 0.01s'

`gate` is the `approve(tool_call)` both backends call, and `policy` builds one from per-tool
modes -- written out here rather than imported from rishi, because the harness needs one
approval shape across both backends and must not stop working because the local engine is
not installed.

In [ ]:
#| export
def always(tool_call): return True
def never(tool_call): return False

def policy(modes, ask):
    """`approve(tool_call)` from per-tool modes: 'approved' | 'check' | 'dont_run'.

    Rishi ships this as `hitl_policy` and fastllm has no equivalent at all, so it is
    written out here rather than imported: the harness needs one approval shape across
    both backends, and it must not stop working because the local engine is not installed.
    `fastllm_hitl.py` is what teaches the cloud side to call it.
    """
    def approve(tc):
        name, _ = _tc(tc)
        mode = (modes or {}).get(name, 'check')
        return True if mode == 'approved' else False if mode == 'dont_run' else ask(tc)
    return approve

In [ ]:
tc = {'function': {'name': 'edit_file', 'arguments': {'path': 'a.py'}}}
bool(gate.gate(tc))

True

In [ ]:
ask_all = Approvals(tools=WRITE_TOOLS, mode='auto').gate
approve = policy({'search_code': 'approved', 'run_python': 'dont_run'}, ask_all)
[bool(approve({'function': {'name': n, 'arguments': {}}})) for n in ('search_code', 'run_python', 'edit_file')]

[True, False, True]

## The fastllm approval shim

Hosted models reach approvals through rishi's own remote path now, so this module is three
functions reporting that there is nothing left to patch. It stays because callers ask.

In [ ]:
#| export
def applied(): return True
def apply(): return True
def note(): return 'provided by rishi.remote'

In [ ]:
applied(), note()

(True, 'provided by rishi.remote')

## Routing a turn before the model sees it

A small deterministic step in front of every turn. Deliberately not another model call:
routing "what default model does leela use?" to the web is exactly the mistake this
prevents, and tool choice should be predictable enough to read in the history.

In [ ]:
#| export
INLINE_SKILLS = ('exhash',)


def tool_plan(prompt):
    """A small deterministic routing step before the model sees a turn.

    This is intentionally not another model call: routing “what default model does this
    project use?” to the web is exactly the mistake this prevents, and tool choice should be
    predictable enough to inspect in conversation history.
    """
    p = str(prompt or '').lower()
    repo = ('this repo', 'repository', 'codebase', 'implementation', 'implemented',
            'default model', 'config', 'source code', 'where is', 'which file', ' method',
            ' function', ' class')
    current = ('latest', 'current docs', 'documentation says', 'release notes', 'on the web',
               'today', 'recent version')
    action = ('create', 'make', 'scale', 'run', 'execute', 'fix', 'change', 'add ', 'remove', 'rename')
    if any(x in p for x in repo):
        return ('repo', 'Use search_code first. Read the matching source if needed. '
                        'Do not web-search a question about the open repository.')
    if any(x in p for x in current):
        return ('web', 'Use web_search, then read_url for the authoritative result.')
    if p.strip().startswith(action) or ' as df_' in p:
        return ('act', 'Use the execution/editing tool that produces the requested result, then verify it.')
    return ('direct', 'Answer directly; use a tool only if the available context is insufficient.')

In [ ]:
[(p, tool_plan(p)[0]) for p in ('where is compaction triggered?',
                                 'what do the latest nbdev release notes say?',
                                 'create a test for threshold',
                                 'why is 2+2 four?')]

[('where is compaction triggered?', 'repo'),
 ('what do the latest nbdev release notes say?', 'web'),
 ('create a test for threshold', 'act'),
 ('why is 2+2 four?', 'direct')]

In [ ]:
test_eq(tool_plan('which file defines Routing?')[0], 'repo')
tool_plan('which file defines Routing?')[1]

'Use search_code first. Read the matching source if needed. Do not web-search a question about Leela or the open repository.'

The frontend composes a message around the question -- the open notebook, the screen -- so
`request_text` recovers the part the person actually typed. Everything that inspects intent
looks at that, not at the envelope.

In [ ]:
#| export
def request_text(prompt):
    """The person's request, excluding notebook/screen context composed around it."""
    text = prompt[-1] if isinstance(prompt, (list, tuple)) and prompt else prompt
    text = str(text or '')
    match = re.search(r'<user-request>\n?(.*?)\n?</user-request>\s*$', text, re.S)
    return match.group(1) if match else text

In [ ]:
request_text('<notebook path=a.ipynb>\n...\n</notebook>\n\n<user-request>\nscale df as df_norm\n</user-request>')

'scale df as df_norm'

A prompt may name a tool or a skill outright. Only names already exposed to the agent
count, so a URL or a path is left alone.

In [ ]:
#| export
def prompt_directives(prompt, tools=(), skills=()):
    """Explicit `/tool` and `/skill-name` mentions embedded in an ordinary prompt.

    Only names already exposed to the agent count, so URLs and filesystem paths are left
    alone. The text after a tool mention is retained as its suggested query; the model
    still chooses arguments for tools whose schema is more structured than one string.
    """
    text = str(prompt or '')
    tool_names = {getattr(t, '__name__', ''): t for t in tools}
    skill_names = {s.name.lower(): s for s in skills}
    matches = list(re.finditer(r'(?<!\S)/([A-Za-z_][\w-]*)', text))
    requested, loaded = [], []
    for i, match in enumerate(matches):
        name = match.group(1)
        if name in tool_names:
            end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
            requested.append((name, text[match.end():end].strip()))
        elif name.lower() in skill_names:
            loaded.append(skill_names[name.lower()])
    return requested, loaded

In [ ]:
host = MemHost({'/proj/a.py': 'def a(): return 1\n'})
ts = tools_for(host)
prompt_directives('/search_code compaction threshold  and see https://x/y/z', tools=ts)

([('search_code', 'compaction threshold  and see https://x/y/z')], [])

## The briefing

The system prompt: what the agent is, where it is, how to work, and what it knows. The
skill index is names and descriptions only -- bodies are a `read_skill` away -- because a
dozen full skill texts would crowd out the code the model is meant to be looking at.

In [ ]:
#| export
MAX_CONTEXT_FILE = 8000     # chars of one AGENTS.md; past this it is documentation, not instructions
CONTEXT_FILES = ('AGENTS.md', '.agents/AGENTS.md', '.leela/AGENTS.md')

def project_context(host, mx=MAX_CONTEXT_FILE):
    """The project's own instructions to an agent, from `AGENTS.md` in each open folder.

    Every other harness reads this file and this one did not, which meant a repository
    could not tell this agent the things it tells every other one -- which package manager
    to use, which directories are generated, how to run the tests. The convention is worth
    more than any rule we could write here, because it is the *project's* rule and it is
    already written down.

    Returned as a block per file with its path on it, so the model can tell a project
    instruction from something the harness made up, and can go read the file itself.
    """
    from pathlib import Path
    out, seen = [], set()
    for r in host.roots or ():
        for rel in CONTEXT_FILES:
            try: p = host.check(Path(r)/rel)
            except Exception: continue
            if str(p) in seen: continue
            seen.add(str(p))
            try: text = host.read(str(p))
            except Exception: text = None
            if not text or not text.strip(): continue
            body = text.strip()
            if len(body) > mx: body = body[:mx] + f'\n…[truncated; read {p} in full if you need the rest]'
            out.append(f'<project_instructions path="{p}">\n{body}\n</project_instructions>')
    if not out: return ''
    return ('\n\n<project_context>\nInstructions this project gives to any agent working in it. They\n'
            'override the general guidance above where they disagree.\n\n' + '\n\n'.join(out) +
            '\n</project_context>')


# The briefing's working rules, each tagged with the tool it is about. A rule for a tool
# the host does not offer is worse than no rule: it advertises a capability that is not
# there, and the model spends turns discovering that. `None` means "always applies".
RULES = (
    (None, 'Act on the user’s verb. “Create”, “run”, “fix”, “add” and “as NAME” request a\n'
           '  result, not a plan: use the tool that produces it, verify it, then report what exists.\n'
           '  Never stop at “I will…”.'),
    (None, 'Never claim a file changed, a command passed, or a test went green unless a tool\n'
           '  result in this conversation says so. If you did not run it, say you did not run it.'),
    (None, 'A tool result starting with ERROR: is a failure. Read it, fix the cause, and try a\n'
           '  different approach. Calling the same tool again unchanged is never the fix.'),
    ('search_code', 'Use `search_code` for project behaviour, unfamiliar APIs, or uncertainty about an\n'
                    '  installed library — the index covers this repo *and* every installed package. Do not\n'
                    '  search for routine Python you already know.'),
    ('grep', 'Use `grep` when you know the exact string: a symbol you are renaming, an error\n'
             '  message, an import. `search_code` finds what is *like* the query; `grep` finds every\n'
             '  place it literally occurs, which is what a rename or an audit needs.'),
    ('view_file', 'Prefer `view_file` over guessing. Copy paths exactly from the workspace, notebook\n'
                  '  context, or search results; never shorten, repair, or reconstruct a path.'),
    ('replace_text', 'To change a file: read it, then `replace_text` with `oldText` copied exactly from\n'
                     '  what you read. Send every change to one file as one call.'),
    ('edit_file', '`edit_file` is the other editor: it addresses lines by the hashes `view_file`\n'
                  '  returns, so a stale read fails instead of damaging the wrong line. Use it when that\n'
                  '  matters, and read the `exhash` skill before your first one.'),
    ('notebook_cells', 'Never use `view_file` or `edit_file` on an `.ipynb` file. Call `notebook_cells` on\n'
                       '  its exact path, choose the returned cell id, then `view_cell` and `edit_cell`.'),
    (None, 'A `<notebook path="…">` block is the open notebook and its exact path. Its cells are\n'
           '  already visible; use that path with the notebook tools and never reconstruct it.'),
    ('run_shell', 'Check your work with `run_shell`: after an edit run the project’s tests, after a\n'
                  '  signature change run its linter or type checker. Use the commands the project itself\n'
                  '  documents (README, pyproject, Makefile). Never start a server, watcher or REPL —\n'
                  '  only commands that exit on their own.'),
    ('run_python', 'Code cells inside `<notebook>` have already executed. Their printed `<output>` is not\n'
                   '  Python and must never be copied into `run_python`. For a request about `df`, call\n'
                   '  `list_vars` first, then run only the transformation the user asked for.'),
    ('run_python', '`run_python` shares the user’s kernel namespace. Read anything; bind results to NEW\n'
                   '  names. You cannot rebind or delete the user’s variables, so do not try.'),
    ('web_search', 'Use `web_search`/`read_url` only when the answer depends on current external\n'
                   '  documentation — not for questions about this repository.'),
    ('memory_search', 'Search remembered research with `memory_search` before paying to search the live web\n'
                      '  again.'),
    ('delegate_parallel', 'When two or more questions are independent and each would take several tool calls,\n'
                          '  send them together with `delegate_parallel` rather than working through them yourself.'),
    (None, 'Make the change the user asked for and no other. Do not reformat, reorganise, or\n'
           '  “improve” code you were not asked to touch, and never discard their edits.'),
    (None, 'Writes may be put to the user for approval. A refusal comes back with their reason —\n'
           '  read it and change the approach, do not retry the same call.'),
    (None, 'Write, edit and run only inside the folders above. Anything else is refused.'),
    (None, 'Be concise. Report what you did and what it cost, not what you intend to do.'),
)


def work_rules(names=()):
    """The briefing's rules, keeping only those whose tool is actually on the table.

    An empty `names` means "do not filter" rather than "no tools": callers that build a
    briefing before the tool list exists (a preview, a test) should still see the whole
    thing.
    """
    names = set(names or ())
    return '\n'.join(f'- {text}' for tool, text in RULES if not names or tool is None or tool in names)


def system_prompt(host, skills=(), inline=INLINE_SKILLS, extra='', tools=()):
    """The agent's briefing: what it is, where it is, how to work, and what it knows.

    The skill index is names and descriptions only -- bodies are a `read_skill` away --
    because a dozen full skill texts would crowd out the code the model is meant to be
    looking at. The exception is spelled out in `INLINE_SKILLS` and is deliberately short.

    `tools` is the tool list the model will actually be given, and it decides which rules
    appear. A briefing assembled independently of the tools drifts from them -- which is
    how this one came to describe a `scale_numeric` that no longer exists, and to promise
    verification the harness had no way to perform.
    """
    names = {getattr(t, '__name__', '') for t in tools or ()}
    roots = '\n'.join(f'  {r}' for r in host.roots) or '  (no folder open)'
    # Only when the host says so. A briefing that promises reads outside the folders on a
    # host that refuses them costs the model a turn discovering that; one that stays silent
    # on a host that allows them costs it the capability.
    if getattr(host, 'read_outside', False):
        roots += ('\n  Reads may name any path on this machine. Writing, running commands and\n'
                  '  listing files stay inside the folders above.')
    # Claimed only where it is true: a host that runs inspections concurrently says so, and
    # telling the model to "keep it short because the kernel is busy" is advice for a problem
    # it may not have.
    conc = ('\n  Your kernel runs each inspection in its own subshell, so this works while one '
            "of the user's cells is still running." if getattr(host, 'concurrent', False) else '')
    live = ('' if 'inspect_python' not in names and names else
            '\n- To *look at* live state, prefer `inspect_python`: neither of its scopes can change\n'
            '  what the user made, so it needs no approval. Start with the default sandbox and pass\n'
            f"  `scope='overlay'` when it refuses a library call you need.{conc}")
    sp = f"""You are Ramabana, a coding agent. Follow the user's latest explicit request and the project instructions below. You are working in these folders:
{roots}

You can search the code index (this repo *and* every installed package), read and edit
files, run commands in the project, run Python in the user's live kernel namespace, read
the web, and read skills that describe the tools already installed here.

How to work:
{work_rules(names)}{live}"""
    idx = skill_index(skills)
    if idx: sp += idx
    for name in inline or ():
        if (s := find(skills, name)): sp += f'\n\n## {s.name}\n\n{s.text()}'
    sp += project_context(host)
    return sp + (f'\n\n{extra}' if extra else '')

In [ ]:
sp = system_prompt(host)
print(sp[:320])

You are Ramabana, a coding agent. Follow the user's latest explicit request and the project instructions below. You are working in these folders:
  /proj

You can search the code index (this repo *and* every installed package), read and edit
files, run commands in the project, run Python in the user's live kernel names


The folders are in it, because a path outside them is refused and the model needs to know
which ones those are. With no folder open it says so rather than leaving the line blank.

In [ ]:
test_eq('/proj' in sp, True)
test_eq('(no folder open)' in system_prompt(NullHost()), True)
len(sp)

4204

## The agent

One `Agent` owns one continuous model context, and everything else hangs off it: routing,
tools, skills, extensions, approvals, compaction, the activity feed and the durable history.
Availability is reported rather than raised -- the model is a multi-gigabyte download on one
side and an API key on the other, and an editor that will not open without either is a worse
editor.

In [ ]:
#| export
class Agent:
    """The IDE's agent: a routed chat whose tools are the host's own capabilities.

    `note` and `ready` report availability rather than raising, the way `Search.backend`
    does: the model is a multi-gigabyte download on one side and an API key on the other,
    and an editor that will not open without either is a worse editor.
    """

    def __init__(self,
                 host,
                 model=None,                # the turn model; None takes the routing default
                 routing=None,
                 sp=None,                   # override the whole briefing
                 approvals=None,            # an `Approvals`; None means nothing is gated
                 cfg=None,                  # config dir, for skills and extensions
                 compact=True,              # compact automatically at the threshold
                 compact_strategy='summary', # 'summary' model checkpoint | 'surgical' deterministic DSL
                 kernel_alive=True,         # what the post-compaction note may promise
                 extensions=True,
                 project_extensions=False,  # project extensions execute repo code: opt in
                 ext_paths=(),
                 inline_skills=INLINE_SKILLS,
                 subagents=True,
                 local_multimodal=False,       # load LiteRT vision/audio encoders for local models
                 tool_max_len=MAX_TOOL_CHARS,
                 on_compact=None,
                 on_activity=None,
                 history_name='agent',      # separate durable conversations can share one config dir
                 poll_every=POLL_EVERY,     # seconds between automatic watch polls; 0 never polls
                 instruction_style='ramabana'): # 'ramabana' | 'aai' compatibility profile
        self.host, self.cfg, self.inline_skills = host, cfg, inline_skills
        if instruction_style not in ('ramabana', 'aai'): raise ValueError('instruction_style must be ramabana or aai')
        self.instruction_style = instruction_style
        self.history_name = history_name
        # One Agent instance owns one continuous model context. Persist that identity on
        # every turn so history can group conversations without pretending separate app
        # launches shared context.
        self.session_id = f'agent_{datetime.datetime.now().strftime("%Y%m%d-%H%M%S")}'
        self.turn_seq, self.current_turn_id = 0, ''
        self.current_branch_id, self.checkpoints = 'main', {}
        self.routing = routing or Routing(turn=model)
        if model: self.routing.set(model)
        self.approvals, self.tool_max_len, self.subagents = approvals, tool_max_len, subagents
        self.local_multimodal = bool(local_multimodal)
        self.extensions, self.project_extensions, self.ext_paths = extensions, project_extensions, ext_paths
        self._sp = sp
        self.compactor = Compactor(auto=compact, strategy=compact_strategy, kernel_alive=kernel_alive, on_compact=on_compact)
        self.activity = Activity(on_change=on_activity)   # the live account of what it is doing
        self.calls = []          # (tool, args) per call this session -- what the UI shows as activity
        self.history = []        # inspectable user/assistant turns, including the chosen tool plan
        self._load_history()
        self.before = {}         # path -> its text just before a write tool touched it, this turn
        self._walked = False     # whether the whole tree was snapshotted for a shell command
        self._tool_calls_turn = 0 # backend-independent guard for local/native tool loops
        self.max_tool_calls = 80
        self.use = Usage()       # this session's total, across every model it routed to
        self.turn_use = Usage()  # the completed foreground turn only; persisted with history
        self._usage_seen = {}    # backend cumulative counters already folded into `use`
        self.note = 'not started'
        self._backends, self._skills, self._reg, self._tools = {}, None, None, None
        self._plain = []         # the unwrapped tools, which is what the briefing is written from
        self.poll_every, self._polled, self._poll_thread = float(poll_every or 0), 0.0, None
        self.lock = threading.Lock()

    # -- what it knows -------------------------------------------------------
    @property
    def skills(self):
        "Every discovered skill, found once. Includes anything an extension registered."
        if self._skills is None:
            try:
                self._skills = discover(self.host.roots, self.cfg, extra=self.registry.skills)
                if self.instruction_style == 'ramabana':
                    self._skills = [s for s in self._skills if s.name != 'coding_patterns']
            except Exception as e:
                self._skills = []
                self.note = f'skills unavailable ({agent_err(e)})'
        return self._skills

    @property
    def registry(self):
        "The extension registry, loaded once. Empty when extensions are switched off."
        if self._reg is None:
            self._reg = Registry(host=self.host, agent=self)
            if self.extensions:
                try: load(self._reg, self.host.roots, self.cfg, self.project_extensions, self.ext_paths)
                except Exception as e: self._reg.notes.append(f'extension loading failed: {agent_err(e)}')
        return self._reg

    @property
    def tools(self):
        "Every tool, built once and recorded. Rebuilt by `reload`."
        if self._tools is None:
            extra = list(self.registry.tools)
            if self.subagents:
                extra += subagent_tools(lambda: self._be_or_none('subagent'), lambda: self._plain)
            plain = tools_for(self.host, lambda: self.skills, extra)
            self._plain = plain
            self._tools = [self._record(t) for t in plain]
        return self._tools

    def reload(self):
        "Re-discover skills, extensions and tools. What a `/reload` command calls after editing them."
        self._skills = self._reg = self._tools = None
        for b in self._backends.values(): b.close()
        self._backends.clear()
        return self

    def refresh(self):
        """Re-discover skills, extensions and tools, and re-brief a running turn backend in place.

        Opening a folder mid-conversation changes what the agent should be told (the roots line
        in the briefing) and may add tools, skills or extensions the new folder carries. Unlike
        `reload`, the loaded engine and the conversation history are kept -- only the system
        prompt and tool set are pushed to a live turn backend, so a folder opened during a chat
        costs nothing on a local model and does not interrupt the conversation.
        """
        self._skills = self._reg = self._tools = None
        spec = self.routing.spec('turn')
        b = self._backends.get((spec.backend, spec.model_id))
        if b is not None: b.refresh(self.system_prompt(), self.tools)
        return self

    def system_prompt(self):
        # `self._plain` rather than `self.tools`: the briefing's rules are chosen by which
        # tools exist, and asking for the recorded wrappers here would build the tool list
        # as a side effect of describing it.
        if self._sp: return self._sp
        if self._tools is None: self.tools
        return system_prompt(self.host, self.skills, self.inline_skills, tools=self._plain)

    # -- recording -----------------------------------------------------------
    def _record(self, f):
        """Wrap one tool so its call is logged and its damage is measurable.

        `functools.wraps` is load-bearing rather than tidy: both backends build their tool
        schema from the function's signature, docstring and annotations, and `__wrapped__`
        is what lets `inspect` see through to the real one. Without it every tool would be
        described to the model as `(*args, **kw)` with no documentation.

        The snapshot has to happen here because this is the last moment the `before` still
        exists. First touch only -- later edits to the same file in one turn are part of
        one change.
        """
        name = getattr(f, '__name__', '?')

        @functools.wraps(f)
        def wrapper(*a, **kw):
            args = _named(f, a, kw)
            self._tool_calls_turn += 1
            if self._tool_calls_turn > self.max_tool_calls:
                return ('Tool-call budget exhausted for this turn. Stop calling tools and '
                        'summarise the evidence and unfinished work now.')
            self.calls.append((name, args))
            meta = self._action_meta(name, args)
            act = self.activity.start(name, args, **meta)
            self.registry.fire('before_tool', self, name, args)
            if name in WRITE_TOOLS:
                if (p := args.get('path')):
                    if p not in self.before: self.before[p] = self.host.text_at(p) or ''
                elif name == 'run_shell': self.snapshot_tree()
            try: out = f(*a, **kw)
            except NotImplementedError as e:
                # The host lost a capability mid-session (a kernel died, a folder closed).
                # A raise here ends the turn; a failure the model can read does not.
                self.activity.finish(act, agent_err(e), ok=False)
                return err(f'{name} is not available here', e)
            except Exception as e:
                self.activity.finish(act, agent_err(e), ok=False)
                raise
            # One spelling of failure, checked in one place. The guess this replaced was a
            # list of prefixes ('edit failed', 'could not') that every new tool had to
            # remember to match, and a tool that phrased its failure any other way was
            # recorded as a success and read by the model as a result.
            self.activity.finish(act, out, ok=not failed(out))
            self.registry.fire('after_tool', self, name, out)
            return out
        return wrapper

    def _action_meta(self, name, args):
        "Frontend-independent identity metadata for a call; applications may override."
        return {'turn_id': self.current_turn_id, 'branch_id': self.current_branch_id}

    def snapshot_tree(self):
        """Read the open folders, so `changes()` can tell what a shell command moved.

        Every other write tool names the file it is about to change, and that one file is
        what gets snapshotted. `run_shell` names nothing. `black .`, a codemod, a `git
        checkout` -- a command can rewrite twenty files and mention none of them, and this
        keyed on a tool argument called `path`, so `changes()` answered `{}`. The README
        leans on `changes()` as the thing that knows what moved, which made the silence a
        good deal worse than the gap.

        The only way to know what a command changed is to have read the files first, so
        this does: once per turn, up to `SHELL_SNAPSHOT` characters. Past that it declines
        and says so, rather than going quiet and returning an empty dict.
        """
        if self._walked: return True
        try: paths = [str(p) for p in self.host.walk()]
        except Exception as e:
            self.host.note(f'cannot watch what commands change: {agent_err(e)}')
            return False
        tree, n = {}, 0
        for p in paths:
            if (text := self.host.text_at(p)) is None: continue
            n += len(text)
            if n > SHELL_SNAPSHOT:
                self.host.note(f'not watching what commands change: the open folders hold over '
                               f'{SHELL_SNAPSHOT // 1_000_000}MB of text')
                return False
            tree[p] = text
        for p, text in tree.items(): self.before.setdefault(p, text)
        self._walked = True
        return True

    def changes(self):
        """`{path: (before, after)}` for every file this turn's write tools actually moved.

        A tool that reported success but changed nothing does not appear here, which is the
        whole point: this is the file, not the claim about the file. A file a shell command
        *created* appears with `''` as its before, exactly as one `create_file` made would.
        """
        out = {}
        for p, was in self.before.items():
            now = self.host.text_at(p)
            if now is not None and now != was: out[p] = (was, now)
        if self._walked:
            # A command can also make files, and a snapshot taken before it ran cannot hold
            # one. Whatever is here now and was not there then is an addition.
            try: paths = [str(p) for p in self.host.walk()]
            except Exception: paths = []
            for p in paths:
                if p in self.before: continue
                if (now := self.host.text_at(p)): out[p] = ('', now)
        return out

    # -- backends ------------------------------------------------------------
    def _be(self, job='turn'):
        """The backend for `job`, built on first use and shared by every job on the same model.

        Shared by model rather than by job on purpose. If summaries and turns happen to
        resolve to the same local model, building two backends would load the engine twice
        -- gigabytes, for no reason. The tools go on whichever backend is the turn model's;
        every other job reaches the engine through `oneshot` or `spawn`, neither of which
        wants them.
        """
        spec = self.routing.spec(job)
        key = (spec.backend, spec.model_id)
        if key not in self._backends:
            is_turn = key == (lambda s: (s.backend, s.model_id))(self.routing.spec('turn'))
            # Rishi defaults multimodal on, which asks LiteRT to construct vision and audio
            # encoders even for text-only bundles such as the local Gemma models. Keep that
            # expensive capability opt-in and pass it to every LiteRT engine, not only the
            # conversational turn backend.
            kw = {'multimodal': self.local_multimodal} if spec.runtime == 'litert' else {}
            if is_turn:
                kw.update(sp=self.system_prompt(), tools=self.tools, tool_max_len=self.tool_max_len,
                          approve=(self.approvals.gate if self.approvals is not None else None))
            self._backends[key] = make_backend(spec, **kw)
        return self._backends[key]

    def _be_or_none(self, job='turn'):
        "The backend for `job` if it can start, else None. What a tool asks, since a tool cannot raise usefully."
        try:
            b = self._be(job)
            return b if b.start() is not None else None
        except Exception: return None

    @property
    def backend(self): return self._be('turn')

    @property
    def chat(self):
        "The live chat object, or None. Kept for the frontends, which use it to cancel."
        return self._be('turn').chat

    @property
    def ready(self):
        """Whether the turn model is up.

        Asked of the backend rather than looked up in the cache. Building a `Backend` is
        just an object -- the expensive part is `start()`, which this does not call -- and
        reaching into `_backends` by key meant anything that supplied a backend some other
        way (a test, a sub-agent harness) reported itself as permanently not ready.
        """
        return self._be('turn').ready

    @property
    def busy(self): return self.lock.locked()

    @property
    def model(self): return self.routing.spec('turn')

    def start(self):
        "Build the turn backend, once. Returns it, or None with `note` explaining why not."
        b = self._be('turn')
        if b.start() is None:
            self.note = b.note
            return None
        # From the backend that is actually running, not from the routing table: a status
        # line that names a model the turn is not on is worse than no status line.
        self.note = f'{model_note(b.spec)} · {len(self.tools)} tools'
        return b

    def retry(self):
        "Forget a previous failure, so a model that has since downloaded or been keyed is picked up."
        b = self._be('turn')
        b.retry()
        return self.start()

    def set_model(self, name, job='turn'):
        "Point `job` at `name`; a turn-model change carries the live conversation with it."
        if self.busy: raise RuntimeError('cannot change model while the assistant is working')
        previous = self.routing.spec(job)
        old = (previous.backend, previous.model_id)
        history = self._backends[old].snapshot_hist() if job == 'turn' and old in self._backends else []
        spec = self.routing.set(name, job)
        new = (spec.backend, spec.model_id)
        if job == 'turn' and new != old:
            self._be('turn').resume_hist(history)
        still_used = {(self.routing.spec(j).backend, self.routing.spec(j).model_id) for j in JOBS}
        if old not in still_used and old in self._backends:
            self._backends.pop(old).close()
        self.note = f'{job} → {model_note(spec)}'
        return spec

    def set_local_multimodal(self, enabled):
        """Choose whether newly loaded LiteRT engines include media encoders.

        The setting belongs to the engine, not a conversation, so changing it releases
        loaded LiteRT backends. They are recreated lazily on the next local request.
        Cloud and llama.cpp backends are unaffected.
        """
        enabled = bool(enabled)
        if enabled == self.local_multimodal: return enabled
        if self.busy: raise RuntimeError('cannot change local multimodal while the assistant is working')
        self.local_multimodal = enabled
        for key in [k for k in self._backends if k[0] == 'litert']:
            self._backends.pop(key).close()
        self.note = f'local multimodal {"on" if enabled else "off"}'
        return enabled

    # -- standing interests --------------------------------------------------
    def poll_watches(self, force=False):
        """Fire whatever the host has due, in a daemon thread, at most every `poll_every` seconds.

        This is the half of the watch feature that was missing. `Host.poll` is the tick the
        whole thing is built around -- a watch is a job somebody arranged to have re-run --
        and nothing ever called it, so a reminder set last week surfaced only if the model
        happened to choose `poll_watches` in some later session. A standing interest that
        depends on being remembered is not standing.

        A turn is the tick, because a turn is the only moment the harness knows somebody is
        here to be reminded. In a thread, because a watch re-reads URLs and the person who
        just pressed enter is not waiting on somebody else's changelog. Whatever fires files
        itself into durable memory on the way past, so the next `memory_search` finds it
        regardless of what this turn was about.
        """
        import time
        if not self.poll_every and not force: return None
        if self._poll_thread is not None and self._poll_thread.is_alive(): return self._poll_thread
        now = time.monotonic()
        if not force and self._polled and now - self._polled < self.poll_every: return None
        self._polled = now

        def run():
            try: r = self.host.poll() or {}
            except NotImplementedError: return           # no watches here; nothing to say about it
            except Exception as e: return self.host.note(f'could not poll watches: {agent_err(e)}')
            if r.get('ran'): self.host.note(f"{r['ran']} of {r.get('checked', 0)} watches fired; see memory_search")
        self._poll_thread = threading.Thread(target=run, name='ramabana-poll', daemon=True)
        self._poll_thread.start()
        return self._poll_thread

    # -- turns ---------------------------------------------------------------
    def _prepare(self, prompt):
        "Everything that happens before a message goes out: notices, hooks, and prospective compaction."
        self.before.clear()                    # `changes()` reports this turn, not the session
        self._walked = False
        self.turn_use = Usage()                # a failed turn cannot inherit the previous turn's cost
        self._tool_calls_turn = 0               # applies even when a native engine owns the loop
        self.turn_seq += 1
        self.current_turn_id = f'{self.session_id}:turn_{self.turn_seq:06d}'
        self.activity.mark(self.current_turn_id) # and so does `turn_md()`
        self.checkpoints[self.current_turn_id] = {'before': self._be('turn').snapshot_hist(),
                                                  'branch_id': self.current_branch_id}
        # Each checkpoint is a deep copy of a whole conversation, so an unbounded dict of them
        # is a session-length memory leak rather than a history feature.
        for old in list(self.checkpoints)[:-MAX_CHECKPOINTS]: self.checkpoints.pop(old, None)
        self.registry.fire('before_turn', self, prompt)
        self.poll_watches()
        outgoing = _with_notices(prompt) if self.instruction_style == 'aai' else prompt
        request = request_text(prompt)
        requested, loaded = prompt_directives(request, self.tools, self.skills)
        route, plan = tool_plan(request)
        if requested:
            route = 'explicit'
            names = ', '.join(dict.fromkeys(name for name, _ in requested))
            plan = f'The user explicitly selected these tools: {names}. Use them before completing the task.'
        self._turn_plan = {'route': route, 'text': plan,
                           'tools': [name for name, _ in requested],
                           'skills': [skill.name for skill in loaded]}
        # Planning has teeth: repo/current-doc routes and explicitly selected safe query
        # tools execute before generation. The model receives evidence instead of merely
        # being advised to choose a tool it may ignore.
        preflights = []
        first = {'repo': 'search_code', 'web': 'web_search'}.get(route)
        if first: preflights.append((first, request))
        eager = {'search_code', 'web_search', 'research', 'memory_search', 'list_files'}
        preflights += [(name, query or request) for name, query in requested if name in eager]
        by_name = {getattr(t, '__name__', ''): t for t in self.tools}
        outgoing = _append(outgoing, f'\n\n<tool-plan route="{route}">{plan}</tool-plan>')
        for name, query in dict.fromkeys(preflights):
            tool = by_name.get(name)
            if tool is None: continue
            try: evidence = tool(query)
            except Exception as e: evidence = f'{name} failed: {agent_err(e)}'
            outgoing = _append(outgoing, f'\n\n<preflight-tool name="{name}">\n{evidence}\n</preflight-tool>')
        for skill in loaded:
            outgoing = _append(outgoing, f'\n\n<requested-skill name="{skill.name}">\n{skill.text()}\n</requested-skill>')
        b = self._be('turn')
        # Looking only at the existing KV cache misses the common failure: a notebook or
        # pasted prompt that crosses the limit in one turn. Measure the actual pending
        # message (including the local chat template) and compact before submitting it.
        if self.compactor.auto and (self.compactor.due(b) or not b.fits(outgoing)): self.compact()
        # Notebook context is complete unless the actual turn cannot fit. Only then apply
        # the per-cell keep/discard policy carried in its markup. Conversation compaction
        # cannot shrink the pending user message, so this has to happen after it.
        if not b.fits(outgoing): outgoing = compact_notebook_context(outgoing, b.fits)
        if not b.fits(outgoing):
            projected = b.projected_tokens(outgoing)
            raise ValueError(f'input is too large for {b.spec.name}: about {projected:,} tokens '
                             f'with a {b.spec.ctx:,}-token context window')
        return outgoing

    @property
    def history_path(self):
        return None if self.cfg is None else self.cfg/f'{self.history_name}-history.jsonl'

    def _load_history(self):
        p = self.history_path
        if p is None or not p.exists(): return
        try: self.history = [json.loads(line) for line in p.read_text().splitlines() if line.strip()][-2000:]
        except Exception: self.history = []

    def sessions(self):
        "Persisted conversations, newest first, with enough detail for a picker."
        grouped = {}
        for turn in self.history:
            sid = turn.get('session') or ''
            if not sid: continue
            grouped.setdefault(sid, []).append(turn)
        return [{'id': sid, 'turns': len(turns), 'at': turns[-1].get('at', 0),
                 'model': turns[-1].get('model', ''),
                 'title': str(turns[0].get('prompt', '')).replace('\n', ' ')[:72]}
                for sid, turns in sorted(grouped.items(), key=lambda x: x[1][-1].get('at', 0), reverse=True)]

    def resume_session(self, selector='latest'):
        "Resume a persisted conversation by full/prefix id, or the newest with `latest`."
        if self.busy: raise RuntimeError('cannot resume while the assistant is working')
        choices = self.sessions()
        if not choices: raise KeyError('no saved sessions; start the CLI with --cfg or use its default config')
        selector = (selector or 'latest').strip()
        if selector == 'latest': picked = choices[0]
        else:
            matches = [s for s in choices if s['id'] == selector or s['id'].startswith(selector)]
            if len(matches) != 1: raise KeyError(f'session {selector!r} matched {len(matches)} conversations')
            picked = matches[0]
        turns = [t for t in self.history if t.get('session') == picked['id']]
        canonical = []
        for turn in turns:
            canonical.append({'role': 'user', 'content': str(turn.get('prompt', ''))})
            if turn.get('reply'): canonical.append({'role': 'assistant', 'content': str(turn['reply'])})
        if picked['model']: self.set_model(picked['model'])
        self._be('turn').resume_hist(canonical)
        self.session_id = picked['id']
        self.note = f"resumed {picked['id']} · {picked['turns']} turns · {picked['model']}"
        return picked

    def _remember(self, prompt, text, error=''):
        turn = {'at': time.time(), 'session': getattr(self, 'session_id', '') or '',
                'turn_id': self.current_turn_id, 'branch_id': self.current_branch_id,
                'model': self._be('turn').spec.name,
                'prompt': str(prompt), 'reply': text, 'error': error,
                'plan': dict(getattr(self, '_turn_plan', {})),
                'usage': self.turn_use.dict(), 'usage_label': repr(self.turn_use),
                'activity': self.activity.rows(mark=self.activity._mark)}
        self.history.append(turn)
        del self.history[:-2000]
        if (p := self.history_path) is not None:
            try:
                p.parent.mkdir(parents=True, exist_ok=True)
                with p.open('a') as f: f.write(json.dumps(turn, ensure_ascii=False) + '\n')
            except Exception: pass

    def _finish(self, text, prompt=''):
        b = self._be('turn')
        # A backend counts cumulatively, so adding `b.use` every turn charges turn one again
        # on turn two. Fold in each backend's *delta* instead, and do it for every backend:
        # a summary or a delegated subagent spends on a routed model that is not `b`.
        turn_use = Usage(model=b.use.model)
        backends = list(self._backends.items())
        if all(backend is not b for _, backend in backends):
            backends.append(((b.spec.backend, b.spec.model_id), b))
        for key, backend in backends:
            previous = self._usage_seen.get(key, Usage(model=backend.use.model))
            turn_use = turn_use + (backend.use - previous)
            self._usage_seen[key] = Usage(**backend.use.dict())
        # Keep the foreground model as the label even when summaries or delegated work
        # added usage on a routed model during this turn.
        turn_use.model = b.use.model or b.spec.model_id
        self.turn_use = turn_use
        self.use = self.use + turn_use
        self.registry.fire('after_turn', self, text)
        if self.current_turn_id in self.checkpoints: self.checkpoints[self.current_turn_id]['after'] = b.snapshot_hist()
        self._remember(prompt, text)
        return text

    def ask(self, prompt, **kw):
        "One turn. Returns the assistant's text, or the reason there isn't any."
        if self.start() is None: return self.note
        with self.lock:
            try:
                outgoing = self._prepare(prompt)
                return self._finish(self._be('turn').send(outgoing, **kw), prompt)
            except Exception as e:
                self.note = f'the assistant failed ({agent_err(e)})'
                self._remember(prompt, self.note, agent_err(e))
                return self.note

    def stream(self, prompt, **kw):
        "One turn as an iterator of markdown chunks, for a frontend that can render as it arrives."
        if self.start() is None:
            yield self.note
            return
        with self.lock:
            try:
                outgoing = self._prepare(prompt)
                out = []
                for chunk in self._be('turn').stream(outgoing, **kw):
                    out.append(chunk)
                    yield chunk
                self._finish(''.join(out), prompt)
            except Exception as e:
                self.note = f'the assistant failed ({agent_err(e)})'
                self._remember(prompt, self.note, agent_err(e))
                yield f'\n\n{self.note}'

    def compose(self, prompt, context='', screen='', image=None, context_path=''):
        """One message from whatever the frontend can supply: the notebook above the
        question, a captured screen as text, and a captured screen as a picture.

        `image` goes in as a *content part* rather than as a tool result, which is both the
        supported path to a multimodal turn on either backend and the right place for it --
        alongside the question asked about it, instead of behind a tool the model has to
        think to call.

        `screen` is the terminal frontend's equivalent and stays text on purpose: a
        terminal screen is a grid of characters, so sending it as prose is both exact and
        far cheaper than a rendering of it would be.

        Separate from `ask_with` so `stream_with` composes identically. A streamed turn
        that quietly saw a different message from a blocking one would be a very hard bug
        to find.
        """
        parts = []
        # A text-only LiteRT engine cannot accept image content parts. Do not hand bytes
        # to it after explicitly omitting the encoders; leave a truthful marker instead.
        local_text_only = bool(image) and self.model.runtime == 'litert' and not self.local_multimodal
        if local_text_only: parts.append('[Image attachment omitted: local multimodal is disabled.]')
        if context:
            # The open notebook's path is operational context, not decoration: without it
            # a small model invents an absolute path before calling view_cell/view_file.
            # It is already sandbox-approved by Workspace.open/Notebook.load.
            attr = f' path="{context_path}"' if context_path else ''
            parts.append(f'<notebook{attr}>\n{context}\n</notebook>')
        if screen: parts.append(f'<screen>\n{screen}\n</screen>')
        parts.append(f'<user-request>\n{prompt}\n</user-request>')
        ask = '\n\n'.join(parts)
        if not image or local_text_only: return ask
        media = list(image) if isinstance(image, (list, tuple)) else [image]
        return [*media, ask]

    def ask_with(self, prompt, context='', screen='', image=None, context_path='', **kw):
        "One turn with the frontend's context attached. Blocking; see `stream_with` for the live one."
        return self.ask(self.compose(prompt, context, screen, image, context_path), **kw)

    def stream_with(self, prompt, context='', screen='', image=None, context_path='', **kw):
        "The same turn, as an iterator of markdown chunks."
        return self.stream(self.compose(prompt, context, screen, image, context_path), **kw)

    def cancel(self):
        "Stop the turn in flight, and release anyone waiting on an approval for it."
        if self.approvals is not None: self.approvals.cancel_all('the turn was stopped')
        return bool(self._be('turn').cancel())

    def close(self):
        for b in list(self._backends.values()):
            try: b.close()
            except Exception: pass
        self._backends.clear()

    def fork(self, turn_id, stage='after', branch_id=''):
        "Fork model context from a captured turn boundary and make it active."
        cp = self.checkpoints.get(str(turn_id))
        if cp is None or stage not in cp: raise KeyError(f'no {stage} checkpoint for {turn_id}')
        branch_id = branch_id or f'branch_{uuid.uuid4().hex[:8]}'
        self._be('turn').restore_hist(cp[stage])
        self.current_branch_id = branch_id
        return {'branch_id': branch_id, 'parent_turn_id': str(turn_id), 'stage': stage}

    def revise(self, turn_id, text, branch_id=''):
        "Fork after a turn and replace its prose response with a user-authored revision."
        branch = self.fork(turn_id, 'after', branch_id)
        self._be('turn').revise_last_assistant(text)
        branch['revision'] = str(text)
        return branch

    # -- the cheap jobs ------------------------------------------------------
    def oneshot(self, prompt, sp='', job='oneshot', max_tokens=None):
        """A question on whichever model `job` routes to, in a conversation that is thrown away.

        `oneshot` is a job in its own right now, so the small fast model every cheap call
        wants has a name that can be pointed somewhere -- `/model qwen-4b oneshot`, or
        `$RAMABANA_MODEL_ONESHOT` -- and `Routing` can find it a substitute when it is not
        installed, instead of the call silently returning nothing.
        """
        b = self._be_or_none(job)
        return '' if b is None else b.oneshot(prompt, sp, max_tokens)

    def classify(self, text, labels):
        "One label for `text`, on the cheap model. Returns the matched label, or the raw reply."
        out = self.oneshot(f'{text}\n\nChoose exactly one label from: {", ".join(labels)}.',
                           'Reply with only the single best label and nothing else.', 'classify', 32).lower()
        return next((l for l in labels if l.lower() in out), out.strip())

    def summarise(self, text, sp='Summarise concisely. Output only the summary.'):
        return self.oneshot(text, sp, 'summary')

    def compact(self, extra=''):
        """Compact the conversation now. Returns the summary text, or `''` with `compactor.note` set.

        The summarizing runs on the `summary` model -- local by default. Paying frontier
        prices to compress a frontier conversation is exactly the spending routing exists
        to stop, and the job is a mechanical transformation of a transcript we already hold.
        """
        b = self._be('turn')
        if b.chat is None:
            self.compactor.note = 'nothing to compact: the model is not running'
            return ''
        sub = self._be_or_none('summary')
        summary_backend = sub if sub is not None else b
        # A 4k local model cannot reserve 4k output tokens after reading a summary prompt.
        # Keep summaries compact and derive their input budget from the model actually used.
        summary_output = min(1024, max(256, summary_backend.spec.ctx // 4))
        summariser = summary_backend.oneshot
        text = self.compactor.compact(
            b, lambda p, sp: summariser(p, sp, summary_output), extra,
            summary_ctx=summary_backend.spec.ctx, summary_output=summary_output,
            summary_count=summary_backend.count_tokens)
        if text: self.registry.fire('compact', self, text)
        self.note = self.compactor.note
        return text

    # -- what to show --------------------------------------------------------
    @property
    def pct_full(self):
        try: return self._be('turn').pct_full
        except Exception: return 0.0

    @property
    def context_used(self):
        try: return self._be('turn').used_tokens
        except Exception: return 0

    def turn_md(self, title='what I did'):
        "This turn's tool calls as foldable markdown, to save alongside the answer in a cell."
        return self.activity.md(mark=self.activity._mark, title=title)

    def turn_lines(self):
        "This turn's tool calls as plain summary lines, for a pane that cannot fold."
        return self.activity.lines(mark=self.activity._mark)

    @property
    def problems(self):
        """Everything that went wrong and had nowhere to be reported, newest last.

        Gathered from every backend rather than kept here, because most of these happen on
        the *cheap* model -- a compaction that could not run, a completion the engine
        refused, an engine complaining on stderr in native code where nothing can catch it.
        Each of those returns `''` to a caller that cannot raise, and until this existed,
        `''` was the whole story the user got.
        """
        out = []
        for b in self._backends.values():
            for p in b.problems:
                if p not in out: out.append(p)
        if (n := self.compactor.note).startswith('compaction') and n not in out: out.append(n)
        return out[-10:]

    def clear_problems(self):
        "Forget them, for a frontend that has shown them."
        for b in self._backends.values(): b.problems.clear()
        return self

    def status(self):
        "Everything a status bar or an `/agent` command wants, in one dict."
        return {'ready': self.ready, 'busy': self.busy, 'note': self.note,
                'problems': self.problems,
                'model': self.model.name, 'model_note': model_note(self.model),
                'ntools': len(self.tools), 'nskills': len(self.skills),
                'pct_full': round(self.pct_full, 3), 'compactions': self.compactor.count,
                'use': self.use.dict(), 'usage': repr(self.use),
                'activity': self.activity.rows(40),
                'approval': (self.approvals.pending.dict() if self.approvals is not None
                             and self.approvals.pending is not None else None),
                'calls': [{'tool': t, 'args': str(a)[:300]} for t, a in self.calls[-40:]]}

    def command(self, line):
        """Run a slash command. Returns text to show, or None when the command is unknown.

        Here rather than in the frontends because both of them need every one of these, and
        a command that exists in the terminal and not the browser is the kind of drift
        `keys.py` was written to prevent.
        """
        line = (line or '').strip().lstrip('/')
        name, _, arg = line.partition(' ')
        arg = arg.strip()
        if name == 'model':
            if not arg: return self.routing.summary()
            job, _, m = arg.partition(' ')
            try: return f'{model_note(self.set_model(m or job, job if m else "turn"))}'
            except Exception as e: return agent_err(e)
        if name == 'sessions':
            rows = self.sessions()
            if rows:
                return '\n'.join(f"{s['id']}  {s['turns']:>3} turns  {s['model']:<20} {s['title']}" for s in rows)
            where = str(self.history_path) if self.history_path is not None else '(history disabled: no cfg directory)'
            return f'no saved sessions in {where}; a session is saved after its first completed turn'
        if name == 'resume':
            try:
                s = self.resume_session(arg or 'latest')
                return f"resumed {s['id']} · {s['turns']} turns · {s['model']}"
            except Exception as e: return agent_err(e)
        if name == 'models':
            rows = available_models(include_legacy=arg.lower() in ('all', 'legacy'))
            if not rows: return 'no models available'
            width = max(len(r['value']) for r in rows)
            return '\n'.join(f"{'*' if r['value'] == self.model.name else ' '} {r['value']:<{width}}  {r['provider']:<12} {r['source']}" for r in rows)
        if name == 'cost': return repr(self.use)
        if name == 'compact':
            t = self.compact(arg)
            return f'{self.compactor.note}\n\n{t}' if t else self.compactor.note
        if name == 'skills':
            return '\n'.join(f'{s.name:16} {s.source:8} {s.description[:90]}' for s in self.skills) or 'no skills found'
        if name == 'skill': return clip(_skill_text(self.skills, arg))
        if name == 'tools': return '\n'.join(sorted(getattr(t, '__name__', '?') for t in self.tools))
        if name == 'extensions': return '\n'.join(self.registry.notes) or 'no extensions loaded'
        if name == 'reload':
            self.reload()
            return f'reloaded: {len(self.tools)} tools, {len(self.skills)} skills'
        if name in self.registry.commands:
            fn, _ = self.registry.commands[name]
            try: return fn(self, arg)
            except Exception as e: return agent_err(e)
        return None

    def commands(self):
        "Every command name, built-in and registered, for a help line or an autocomplete."
        return sorted({'model', 'models', 'sessions', 'resume', 'cost', 'compact', 'skills', 'skill', 'tools', 'extensions', 'reload',
                       *self.registry.commands})

In [ ]:
#| export
def _named(f, a, kw=None):
    """Every argument as a dict, positional ones matched to their parameter names.

    Models normally send named arguments, but an extension's tool may be called
    positionally by another extension -- and it may be called *both* ways at once, which is
    where this used to go wrong. The caller asked for `kw if kw else _named(f, a)`, so one
    keyword argument threw the whole positional half away: `edit_file('a.py', commands=...)`
    was recorded as `{'commands': ...}`. That is an activity line with no path in it, and
    worse, `changes()` looks for the path here -- so the file was never snapshotted and the
    edit never showed up as a change.
    """
    kw = dict(kw or {})
    if not a: return kw
    try:
        import inspect
        return {**dict(zip(list(inspect.signature(f).parameters), a)), **kw}
    except Exception: return {**{'args': a}, **kw}


def _append(prompt, text):
    """Add `text` to a message that may be a list of content parts rather than a string.

    `compose` returns a list when an image is attached, and `list += str` extends the list
    one character at a time -- so a turn with a screenshot arrived at the model as the image
    followed by several hundred single-character parts, with the tool plan and every
    preflight result shredded among them. The text belongs on the message's text part.
    """
    if not isinstance(prompt, (list, tuple)): return prompt + text
    parts = list(prompt)
    for i in range(len(parts) - 1, -1, -1):
        if isinstance(parts[i], str):
            parts[i] += text
            return parts
    return [*parts, text]


def _skill_text(skills, name):
    s = find(skills, name)
    return f'no skill matching {name!r}' if s is None else s.text()


def _with_notices(prompt):
    "Append the aai-coding style prompt notices, when the prompt earns any. Text prompts only."
    if not isinstance(prompt, str): return prompt
    return prompt + notices_block(prompt)

`fake_agent` from [testing](04_testing.ipynb) is a real `Agent` whose every job routes to a
scripted backend, so a turn can be driven here with no model at all.

In [ ]:
agent, be = fake_agent(host, replies=['`threshold` is in `ramabana/runtime.py`.'])
agent.ask('where is the compaction threshold?')

'`threshold` is in `ramabana/runtime.py`.'

The turn was routed, and the route is recorded rather than inferred: this prompt asked
about the repository, so the plan said search first.

In [ ]:
agent._turn_plan

Planning has teeth. A repo question runs `search_code` *before* generation and hands the
model the evidence, instead of advising it to choose a tool it may ignore -- which is what
the composed message shows:

In [ ]:
print(be.sent[0][-400:])

 question mark, so it is a question. Make only the tool calls needed to answer it, then answer it, then stop -- do not start the work it implies.
</system-reminder>

<tool-plan route="repo">Use search_code first. Read the matching source if needed. Do not web-search a question about Leela or the open repository.</tool-plan>

<preflight-tool name="search_code">
no matches (memory)
</preflight-tool>


Every call is recorded as it happens, so the feed and the saved cell agree with each other.

In [ ]:
agent.calls, agent.turn_lines()

([('search_code', {'query': 'where is the compaction threshold?'})],
 ['🔍 Search where is the compaction threshold?'])

`status()` is everything a status bar or an `/agent` command wants, in one dict.

In [ ]:
{k: v for k, v in agent.status().items() if k in ('ready', 'busy', 'model', 'ntools', 'pct_full', 'usage')}

{'ready': True,
 'busy': False,
 'model': 'ornith-9b',
 'ntools': 14,
 'pct_full': 0.015,
 'usage': '15 tok · in 10 · out 5 · model'}

A write tool is snapshotted before it runs, so `changes()` reports the file rather than the
claim about the file -- a tool that said it succeeded and changed nothing does not appear.

In [ ]:
agent.ask('create a file')
create = next(t for t in agent.tools if t.__name__ == 'create_file')
create('/proj/b.py', 'def b(): return 2\n')
agent.changes()

{'/proj/b.py': ('', 'def b(): return 2\n')}

Slash commands live here rather than in the frontends, because both of them need every one
of these and a command that exists in the terminal and not the browser is exactly the drift
worth preventing. An unknown command returns `None`, which is how a frontend knows to pass
it on.

In [ ]:
agent.command('/tools').splitlines()[:4]

['create_file', 'create_skill', 'delegate_parallel', 'delegate_search']

In [ ]:
test_eq(agent.command('/nope'), None)
print(agent.command('/model'))

turn        ornith-9b · local · 32k ctx
inline      ornith-9b · local · 32k ctx
completion  qwen-4b · local · 32k ctx
classify    qwen-4b · local · 32k ctx
summary     qwen-4b · local · 32k ctx
subagent    ornith-9b · local · 32k ctx


The cheap jobs route away from the turn model: a classification or a summary runs on
whatever `classify` and `summary` point at, in a conversation that is thrown away.

In [ ]:
agent.oneshot('is this a question?'), agent.summarise('a long transcript')

('ONESHOT:is this a question?', 'ONESHOT:a long transcript')

Compaction is one call on the summary model, and it reports what it did either way.

In [ ]:
agent.compact(), agent.compactor.note

('ONESHOT:<conversation>\n<message index=1 role=use',
 'compacted 4 message(s), kept 4')

Problems are gathered from every backend rather than kept on the agent, because most of them
happen on the *cheap* model -- a compaction that could not run, a completion the engine
refused -- where the caller cannot raise and used to get `''` as the whole story.

In [ ]:
agent.problems, agent.pct_full

([], 0.015)

## Inline completion

Inline completion is the argument for routing in one feature: four lines of code, fired
constantly, has to feel instant, worth approximately nothing per call. It runs on the local
model in a throwaway conversation, and never automatically -- the kernel's own completions
are instant and correct, while this costs a forward pass.

In [ ]:
#| export
COMPLETE_SP = """You are a code completion engine inside an editor. You are given the code before \
the cursor in <before> and the code after it in <after>.

Reply with ONLY the code that belongs at the cursor. No explanation, no markdown fence, no \
repetition of <before> or <after>. Keep it short -- finish the current expression, statement or \
short block and stop. Match the surrounding indentation and style exactly. If nothing sensible \
belongs there, reply with nothing at all."""

MAX_COMPLETION_LINES = 4     # a suggestion longer than this is a guess about the design, not a completion
COMPLETION_TOKENS = 96
CTX_BEFORE, CTX_AFTER = 2000, 600   # chars of surrounding code sent as context


def _strip_echo(before, out):
    "Drop a re-emitted tail of `before` from the front of `out` -- models like to restate the line they continue."
    tail = before[-200:]
    for n in range(len(tail), 0, -1):
        if out.startswith(tail[-n:]): return out[n:]
    return out

In [ ]:
#| export
def _fence_tail(text):
    """What follows an *unterminated* fence, or None.

    A small model asked for bare code often explains itself first and then opens a fence, and
    the token cap arrives before the closing one. Everything before that opener is prose, so
    inserting it would put an essay in the middle of the user's file.
    """
    if '```' not in text: return None
    head, _, rest = text.rpartition('```')
    if '```' in head and head.count('```') % 2: return None   # a complete block: fenced_blocks has it
    return rest.partition('\n')[2] if '\n' in rest else ''


def _clean(text, before, max_lines):
    """A raw model reply as something safe to insert: fences off, prose off, echo off, `max_lines` long.

    `fenced_blocks` rather than a ```python-only matcher because a completion model asked for
    bare code and fencing it anyway rarely bothers with an info string. It also keeps this
    function testable without a model installed.
    """
    from fastcore.xtras import fenced_blocks
    text = text or ''
    if (blocks := fenced_blocks(text)): out = blocks[-1][1]
    elif (tail := _fence_tail(text)) is not None: out = tail
    else: out = text
    out = _strip_echo(before, (out or '').strip('\n'))
    lines = out.split('\n')[:max_lines]
    while lines and not lines[-1].strip(): lines.pop()
    return '\n'.join(lines)

`_clean` is what makes a raw reply safe to insert. A completion model asked for bare code
fences it anyway, restates the line it is continuing, and keeps going past the point where
it is guessing about the design.

In [ ]:
before = 'def threshold(ctx, reserve=RESERVE):\n    '
_clean('```\n    if not ctx: return None\n    return max(1, ctx - reserve)\n```', before, 4)

'if not ctx: return None\n    return max(1, ctx - reserve)'

In [ ]:
test_eq(_clean('    ' + 'x = 1', '    ', 4), 'x = 1')       # the echoed indent is dropped
_clean('a\nb\nc\nd\ne\nf', '', 3)

'a\nb\nc'

The shape a small local model actually produces is prose, then a fence the token cap cut off
before it closed. Everything before the opener is the essay, and inserting *that* into the
buffer is the one outcome worse than suggesting nothing.

In [ ]:
reply = 'Looking at the code, `area(r)` returns nothing. Let me fix it:\n```python\n    return math.pi * r**2'
_clean(reply, before, 4)

'return math.pi * r**2'

In [ ]:
test_eq(_clean('I will explain first.\n```python', before, 4), '')   # nothing but prose: suggest nothing
_clean('```\ncomplete block\n```', '', 4)

'complete block'

In [ ]:
#| export
class Completer:
    """Inline completion, always on the local model.

    Routing sends `completion` to the local backend unconditionally, and that is the whole
    argument for routing in one feature: a completion is four lines of code, fires
    constantly, has to feel instant, and is worth approximately nothing per call. Sending
    it to a frontier model would be slower, cost real money, and put every keystroke's
    surroundings on someone else's wire.

    A completion is a *question about the code on screen*, not a turn in a conversation, so
    it runs in a throwaway conversation on an engine that is already loaded. Letting
    suggestions accumulate would poison the assistant's context as well as their own.

    Nothing here is automatic. The kernel's own completions are instant and correct and
    stay bound to typing; this costs a forward pass, so it fires only when asked for.
    """

    def __init__(self, agent, max_lines=MAX_COMPLETION_LINES, max_tokens=COMPLETION_TOKENS):
        self.a, self.max_lines, self.max_tokens = agent, max_lines, max_tokens
        self.note = 'not asked yet'

    @property
    def ready(self):
        b = self.a._be_or_none('completion')
        return b is not None

    def _prompt(self, code, pos, lang, context=''):
        try: variables = self.a.host.list_vars()
        except Exception: variables = ''
        support = ''
        if context: support += f'<related_code>\n{context[-6000:]}\n</related_code>\n'
        if variables: support += f'<runtime_variables>\n{variables[:4000]}\n</runtime_variables>\n'
        # Whatever the application has pinned to completion. Duck-typed rather than an
        # interface: the harness has no opinion about where a note came from, and a host
        # that keeps none simply has no `ws`.
        try: memory = self.a.ws.agent_memory_context('completion', max_chars=6000)
        except Exception: memory = ''
        if memory: support += f'<user_memory>\n{memory}\n</user_memory>\n'
        return (f'Language: {lang}\n\n{support}<before>\n{code[:pos][-CTX_BEFORE:]}\n</before>\n'
                f'<after>\n{code[pos:][:CTX_AFTER]}\n</after>')

    def complete(self, code, pos, lang='python', context=''):
        "The text to insert at `pos` in `code`, or `''` with `note` saying why there isn't any."
        b = self.a._be_or_none('completion')
        if b is None:
            self.note = 'no completion model available'
            return ''
        if b.busy:
            # One engine means one generation at a time. Declining beats queueing behind a
            # tool loop the user is watching: "the model is thinking" is a better answer
            # than an editor that has stopped taking keys.
            self.note = 'model busy — it is mid-turn'
            return ''
        text = b.oneshot(self._prompt(code, pos, lang, context), COMPLETE_SP, self.max_tokens)
        if not text:
            self.note = b.note if not b.ready else 'no suggestion'
            return ''
        out = _clean(text, code[:pos], self.max_lines)
        self.note = f'{len(out.splitlines())} line(s) from {b.spec.name}' if out else 'no suggestion'
        return out

The completer asks the `completion` backend, and says why there is nothing when there is
nothing -- an editor that silently inserts nothing is indistinguishable from a broken one.

In [ ]:
comp = Completer(agent)
comp.ready, comp.complete('def a():\n    ', 12)

(True, 'ONESHOT:Language: python\n\n<before>\ndef a():')

In [ ]:
comp.note

'4 line(s) from fake'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()